## TRIVAL Stage 2

This code is provided as an example code to implement Thematic Review with inbuilt validation (TRIVAL) Stage 2. The users are free to download and make the changes based on their requirement. This exercise tests the sequence entailment approach. The idea is to validate if the themes identified in stage 1 truly aligns with the abstract. In order to test it, we pass all the themes and abstract to BART and sentence bert (two different LLMs).

We experiment with two different abstracts and find convergnce. i.e the theme for the abstract got maxmimum score in both the cases (with multi-label as true and false). A multi label true means, the model evaluates each theme for a given abstract independently.

To execute the code. the user has to create profile at huggingface (https://huggingface.co/) and generate an access token (API key). The user will not be able to execute the code chunks without it.

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os

## HF_API_KEY need to be replaced with user genereated API key at hugging face
os.environ["HF_TOKEN"] = userdata.get("HF_API_KEY")
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Import Data

There are several ways to import the file. However, we will save the file in google drive and read the file in this notebook. Once the files are uploaded, the below code chunk helps mount the google drive for colab use. It will ask for permission to access google drive. Users must grant permission by giving their signin credentials.

In [2]:
from google.colab import drive
drive.mount('/content/files')
#drive.flush_and_unmount()

Mounted at /content/files


We create a folder in googel drive named "BertWithLlama2" inside which the TRIVALStage1.ipnb and TRIVALStage2.ipynb files is available along with the file that contains abstract and other metadata associated with it. We have named this file as "rawdata694_08082023.csv". Users are free to rename it. Once the access is provided, the files can be accessed by giving the pathname as "/content/files/MyDrive/BertWithLlama2/"

In [3]:
path_name = "/content/files/MyDrive/BertWithLlama2/"
#to change the dir to path_name
os.chdir(path_name)
#to re-print the dir name
os.getcwd()

'/content/files/MyDrive/BertWithLlama2'

We are using pandas library to read the files. So we import pandas package using the below code line and then use read_csv() to import the rawdata694_08082023.csv. The sccessful read of the file will show the first five rows of information from the file.

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('./After First Manual Sorting_V1.1.csv', na_values =['[No abstract available]', ''])
df.head()

,Unique ID,Article Title,Source Title,Abstract,Publication Date,Publication Year,Broad Single Category,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,1.0,Late marriage as a contributor to the industri...,ECONOMIC HISTORY REVIEW,Was the European marriage pattern an important...,NOV,2018.0,Marriage Delay,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,Late marriage and the HIV epidemic in sub-Saha...,POPULATION STUDIES-A JOURNAL OF DEMOGRAPHY,The causes of large variation in the sizes of ...,MAR,2007.0,Marriage Delay,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,WELFARE REFORM AND LONE PARENTS IN THE UK,ECONOMIC JOURNAL,A series of reforms to help low income familie...,FEB,2009.0,Single Parent,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,Lone parents and informal childcare: A tax cre...,SOCIAL POLICY & ADMINISTRATION,The Labour government aims to increase the lon...,DEC,2006.0,Single Parent,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,Labor supply in the terminal stages of lone pa...,JOURNAL OF POPULATION ECONOMICS,"Based on Norwegian register data, we show that...",OCT,2012.0,Single Parent,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 997 entries, 0 to 996
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unique ID              944 non-null    float64
 1   Article Title          944 non-null    object 
 2   Source Title           944 non-null    object 
 3   Abstract               933 non-null    object 
 4   Publication Date       891 non-null    object 
 5   Publication Year       944 non-null    float64
 6   Broad Single Category  923 non-null    object 
 7   Unnamed: 7             0 non-null      float64
 8   Unnamed: 8             0 non-null      float64
 9   Unnamed: 9             0 non-null      float64
 10  Unnamed: 10            0 non-null      float64
 11  Unnamed: 11            0 non-null      float64
 12  Unnamed: 12            0 non-null      float64
 13  Unnamed: 13            0 non-null      float64
 14  Unnamed: 14            0 non-null      float64
 15  Unname

In [6]:
# Extract abstracts to train on and corresponding titles
titles = df["Article Title"].fillna("")  # Fill NaN values with empty string
abstracts = df["Abstract"].fillna("") # Fill NaN values with empty string

In [9]:
print(abstracts[120])

The purpose of this study is to examine the effects of race, gender, and marital status on giving and volunteering behavior. A second purpose is to examine these effects across different survey methodologies. Using data from Indiana households, a multimethod, multigroup research design was used to compare giving and volunteering across eight. different survey methodologies. Results indicate important differences in philanthropic behaviors by gender, race, marital status, and survey methodology-even when controlling for differences in income, age, and educational attainment. These results highlight the importance of looking specifically at human and social capital variables, and survey methodology, when making assumptions about and interpreting the measurement of philanthropic behavior.


### Classifying each abstract one by one using BART

Test on one sample

We use BART model from META AI to test the sequence entailment problem. Users may read more about the model at: https://huggingface.co/facebook/bart-large-mnli. With the advancement happening in deep learing/GenAI, several new models are relased every monnth. Users can check the list of new ZSL models here: https://huggingface.co/models?pipeline_tag=zero-shot-classification&sort=trending

In [7]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [23]:
## The correct label to below premise is

sequence_to_classify = """The purpose of this study is to examine the effects of race, gender, and marital status on giving and volunteering behavior.
A second purpose is to examine these effects across different survey methodologies. Using data from Indiana households, a multimethod, multigroup research
design was used to compare giving and volunteering across eight. different survey methodologies. Results indicate important differences
in philanthropic behaviors by gender, race, marital status, and survey methodology-even when controlling for differences in income, age, and educational attainment.
These results highlight the importance of looking specifically at human and social capital variables, and survey methodology,
when making assumptions about and interpreting the measurement of philanthropic behavior.
"""

With the correct label, we also pass the categories in the variable candidate_labels. In the below code, users can provide their own list of classes in which they will like to classify a text. This implies that supervised learning techiques listed in computational literature review method can be easily implemented using this stage of the framework (without the need for any model training).

In [31]:
candidate_labels = ['The article uses qualitative research method',
                    'The article uses quantitative research method',
                    'The article uses mixed method for study'
                    ]


If more than one candidate label can be correct, pass multi_label=True to calculate each class independently.

In [33]:
classifier(sequence_to_classify, candidate_labels, multi_label=True)

{'sequence': 'The purpose of this study is to examine the effects of race, gender, and marital status on giving and volunteering behavior. \nA second purpose is to examine these effects across different survey methodologies. Using data from Indiana households, a multimethod, multigroup research \ndesign was used to compare giving and volunteering across eight. different survey methodologies. Results indicate important differences \nin philanthropic behaviors by gender, race, marital status, and survey methodology-even when controlling for differences in income, age, and educational attainment. \nThese results highlight the importance of looking specifically at human and social capital variables, and survey methodology, \nwhen making assumptions about and interpreting the measurement of philanthropic behavior.\n',
 'labels': ['The article uses mixed method for study',
  'The article using quantitative research method',
  'The article using qualitative research method'],
 'scores': [0.96

### Classification using BART on full data

Classify the entire corpus of abstract using the candidate_labels

In [34]:
results_bart = []
for i, abstract in enumerate(abstracts):
    if abstract:
        print(f"\nProcessing abstract {i+1}:\n{abstract[:150]}...") # Print a snippet of the abstract
        # Re-assign sequence_to_classify for each abstract
        sequence_to_classify = abstract
        # Perform classification
        classification_output = classifier(sequence_to_classify, candidate_labels, multi_label=True)
        results_bart.append({"abstract_index": i, "abstract_snippet": abstract[:150], "classification": classification_output})
    else:
        print(f"\nSkipping empty abstract at index {i+1}")

# Display results for each abstract
for res in results_bart:
    print(f"\nAbstract Index: {res['abstract_index']}")
    print(f"Abstract Snippet: {res['abstract_snippet']}...")
    print("Classification Results:")
    for label, score in zip(res['classification']['labels'], res['classification']['scores']):
        print(f"  - {label}: {score:.4f}")

Streaming output truncated to the last 5000 lines.
  - The article using qualitative research method: 0.9174
  - The article using quantitative research method: 0.8048

Abstract Index: 219
Abstract Snippet: Since June 26, 2015. marriages to same-sex couples have been legally recognized across every state in the United States, bringing new challenges to me...
Classification Results:
  - The article uses mixed method for study: 0.9796
  - The article using quantitative research method: 0.7371
  - The article using qualitative research method: 0.5822

Abstract Index: 220
Abstract Snippet: Non-marital birth rates for women in their twenties began declining in 2008, but the mechanisms driving this decline are not yet well understood. Usin...
Classification Results:
  - The article using quantitative research method: 0.8332
  - The article uses mixed method for study: 0.6215
  - The article using qualitative research method: 0.3526

Abstract Index: 221
Abstract Snippet: We explore trends in 

### Saving BART Classification Results to CSV

The CSV file `abstract_classification_results.csv` will be created and saved with the article titles, abstracts, and their corresponding classification scores for each label. You can find it in your current working directory after running the below code

In [36]:
import pandas as pd

output_data = []

# Prepare column names for the classification scores dynamically
score_columns = [f"Score_{label}" for label in candidate_labels]

for res in results_bart:
    abstract_idx = res['abstract_index']
    article_title = titles.iloc[abstract_idx]
    full_abstract = abstracts.iloc[abstract_idx]

    # Initialize scores for all candidate labels to NaN or 0.0
    scores_dict = {col: 0.0 for col in score_columns}

    # Fill in actual scores from the classification output
    classification_labels = res['classification']['labels']
    classification_scores = res['classification']['scores']

    for label, score in zip(classification_labels, classification_scores):
        scores_dict[f"Score_{label}"] = score

    row_data = {
        "Article Title": article_title,
        "Abstract": full_abstract
    }
    row_data.update(scores_dict)
    output_data.append(row_data)

# Create a DataFrame from the processed data
output_df = pd.DataFrame(output_data)

# Save the DataFrame to a CSV file
output_filename = "abstract_method_classification_results_bart.csv"
output_df.to_csv(output_filename, index=False)

print(f"Classification results saved to {output_filename}")
display(output_df.head())

Classification results saved to abstract_classification_results_method.csv


,Article Title,Abstract,Score_The article using qualitative research method,Score_The article using quantitative research method,Score_The article uses mixed method for study
0,Late marriage as a contributor to the industri...,Was the European marriage pattern an important...,0.412837,0.697295,0.995243
1,Late marriage and the HIV epidemic in sub-Saha...,The causes of large variation in the sizes of ...,0.385897,0.645536,0.972846
2,WELFARE REFORM AND LONE PARENTS IN THE UK,A series of reforms to help low income familie...,0.416256,0.483880,0.911787
3,Lone parents and informal childcare: A tax cre...,The Labour government aims to increase the lon...,0.995848,0.012021,0.996764
4,Labor supply in the terminal stages of lone pa...,"Based on Norwegian register data, we show that...",0.118431,0.603177,0.542779


## Classification using SBERT

Earlier we used BART for ZSL. Here we use SBERT for scoring the text and different classes. Intutively, think of this model as another individual who in parallel is trying to validate the result. The outcome from BART and SBERT can then we used to come to a consensus (expert judgement, intercoder reliability).

In [37]:
# load the sentence-bert model from the HuggingFace model hub
#!pip install transformers
from transformers import AutoTokenizer, AutoModel
from torch.nn import functional as F
tokenizer = AutoTokenizer.from_pretrained('deepset/sentence_bert')
model = AutoModel.from_pretrained('deepset/sentence_bert')

sentence = sequence_to_classify
labels = candidate_labels

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

### Classifying the entire corpus of abstracts using SBERT

In [38]:
import pandas as pd

# Assuming output_df was created and contains the BART classification results
# If not, the code to generate output_df would need to be re-run.

# Check if output_df exists from previous execution
if 'output_df' in locals() or 'output_df' in globals():
    display(output_df.head())
else:
    print("output_df not found. It might not have been generated yet or the kernel was reset.")
    print("Please run the cell that generates 'abstract_classification_results.csv' if you haven't already.")
    # Optionally, load from CSV if preferred, but running the generating cell is more direct.
    # output_df = pd.read_csv("abstract_classification_results.csv")
    # display(output_df.head())


,Article Title,Abstract,Score_The article using qualitative research method,Score_The article using quantitative research method,Score_The article uses mixed method for study
0,Late marriage as a contributor to the industri...,Was the European marriage pattern an important...,0.412837,0.697295,0.995243
1,Late marriage and the HIV epidemic in sub-Saha...,The causes of large variation in the sizes of ...,0.385897,0.645536,0.972846
2,WELFARE REFORM AND LONE PARENTS IN THE UK,A series of reforms to help low income familie...,0.416256,0.483880,0.911787
3,Lone parents and informal childcare: A tax cre...,The Labour government aims to increase the lon...,0.995848,0.012021,0.996764
4,Labor supply in the terminal stages of lone pa...,"Based on Norwegian register data, we show that...",0.118431,0.603177,0.542779


In [40]:
import torch
from torch.nn import functional as F

results_sbert = []

for i, abstract_text in enumerate(abstracts):
    if abstract_text:
        print(f"\nProcessing abstract {i+1}:\n{abstract_text[:150]}...")

        # Prepare inputs for SBERT
        inputs = tokenizer.batch_encode_plus([abstract_text] + candidate_labels,
                                             return_tensors='pt',
                                             padding=True
                                             )
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        # Run inputs through model and mean-pool over the sequence dimension
        with torch.no_grad(): # Disable gradient calculation for inference
            output = model(input_ids, attention_mask=attention_mask)[0]

        sentence_rep = output[:1].mean(dim=1)
        label_reps = output[1:].mean(dim=1)

        # Calculate cosine similarities
        similarities = F.cosine_similarity(sentence_rep, label_reps)

        # Sort labels by similarity score in descending order
        sorted_indices = similarities.argsort(descending=True)

        # Store classification output
        classification_output_sbert = {
            'labels': [candidate_labels[idx] for idx in sorted_indices],
            'scores': [similarities[idx].item() for idx in sorted_indices]
        }

        results_sbert.append({"abstract_index": i, "abstract_snippet": abstract_text[:150], "classification": classification_output_sbert})
    else:
        print(f"\nSkipping empty abstract at index {i+1}")

# Display results for each abstract
for res in results_sbert:
    print(f"\nAbstract Index: {res['abstract_index']}")
    print(f"Abstract Snippet: {res['abstract_snippet']}...")
    print("Classification Results:")
    for label, score in zip(res['classification']['labels'], res['classification']['scores']):
        print(f"  - {label}: {score:.4f}")

Streaming output truncated to the last 5000 lines.
  - The article using qualitative research method: 0.1243
  - The article using quantitative research method: 0.0696

Abstract Index: 219
Abstract Snippet: Since June 26, 2015. marriages to same-sex couples have been legally recognized across every state in the United States, bringing new challenges to me...
Classification Results:
  - The article uses mixed method for study: 0.2454
  - The article using qualitative research method: 0.2168
  - The article using quantitative research method: 0.1784

Abstract Index: 220
Abstract Snippet: Non-marital birth rates for women in their twenties began declining in 2008, but the mechanisms driving this decline are not yet well understood. Usin...
Classification Results:
  - The article using qualitative research method: 0.1325
  - The article uses mixed method for study: 0.1087
  - The article using quantitative research method: 0.0618

Abstract Index: 221
Abstract Snippet: We explore trends in 

### Saving SBERT Classification Results to CSV

The CSV file `abstract_classification_results_sbert.csv` will be created and saved with the article titles, abstracts, and their corresponding Sentence-BERT classification scores for each label. You can find it in your current working directory after running the below code

In [41]:
import pandas as pd
import torch # Import torch to use its no_grad context manager

output_data_sbert = []

# Prepare column names for the classification scores dynamically
score_columns_sbert = [f"SBERT_Score_{label}" for label in candidate_labels]

for res in results_sbert:
    abstract_idx = res['abstract_index']
    article_title = titles.iloc[abstract_idx]
    full_abstract = abstracts.iloc[abstract_idx]

    # Initialize scores for all candidate labels to 0.0
    scores_dict_sbert = {col: 0.0 for col in score_columns_sbert}

    # Fill in actual scores from the classification output
    classification_labels_sbert = res['classification']['labels']
    classification_scores_sbert = res['classification']['scores']

    for label, score in zip(classification_labels_sbert, classification_scores_sbert):
        scores_dict_sbert[f"SBERT_Score_{label}"] = score

    row_data_sbert = {
        "Article Title": article_title,
        "Abstract": full_abstract
    }
    row_data_sbert.update(scores_dict_sbert)
    output_data_sbert.append(row_data_sbert)

# Create a DataFrame from the processed data
output_df_sbert = pd.DataFrame(output_data_sbert)

# Save the DataFrame to a CSV file
output_filename_sbert = "abstract_method_classification_results_sbert.csv"
output_df_sbert.to_csv(output_filename_sbert, index=False)

print(f"SBERT Classification results saved to {output_filename_sbert}")
display(output_df_sbert.head())

SBERT Classification results saved to abstract_method_classification_results_sbert.csv


,Article Title,Abstract,SBERT_Score_The article using qualitative research method,SBERT_Score_The article using quantitative research method,SBERT_Score_The article uses mixed method for study
0,Late marriage as a contributor to the industri...,Was the European marriage pattern an important...,0.105966,0.081871,0.006999
1,Late marriage and the HIV epidemic in sub-Saha...,The causes of large variation in the sizes of ...,0.115446,0.059316,0.039297
2,WELFARE REFORM AND LONE PARENTS IN THE UK,A series of reforms to help low income familie...,0.111499,0.063284,0.042538
3,Lone parents and informal childcare: A tax cre...,The Labour government aims to increase the lon...,0.140571,0.058368,0.105630
4,Labor supply in the terminal stages of lone pa...,"Based on Norwegian register data, we show that...",-0.078250,-0.037990,-0.102693


### Identify Country Names from Abstracts using spaCy

In [28]:
# Install spaCy
!pip install spacy

# Download a pre-trained English model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [29]:
import spacy

# Load the English language model
# You might need 'en_core_web_md' or 'en_core_web_lg' for better accuracy,
# but 'en_core_web_sm' is a good starting point and lighter.
nlp = spacy.load('en_core_web_sm')

identified_countries = []

for i, abstract in enumerate(abstracts):
    if abstract:
        doc = nlp(abstract)
        countries_in_abstract = []
        for ent in doc.ents:
            # 'GPE' (Geopolitical Entity) often includes countries, cities, states.
            # We'll filter for entities that are likely to be countries.
            if ent.label_ == 'GPE' and ent.text[0].isupper() : # Simple heuristic for country names
                countries_in_abstract.append(ent.text)

        # Remove duplicates and store
        unique_countries = list(set(countries_in_abstract))
        identified_countries.append({
            "abstract_index": i,
            "abstract_snippet": abstract[:150],
            "countries": unique_countries
        })
        print(f"\nProcessing abstract {i+1}: Found countries: {unique_countries}")
    else:
        identified_countries.append({
            "abstract_index": i,
            "abstract_snippet": "",
            "countries": []
        })
        print(f"\nSkipping empty abstract at index {i+1}")

# Display some of the results
print("\n--- Identified Countries Summary ---")
for res in identified_countries[:10]: # Displaying first 10 for brevity
    print(f"Abstract Index: {res['abstract_index']}, Countries: {res['countries']}")





Processing abstract 1: Found countries: ['England']

Processing abstract 2: Found countries: ['Kenya', 'Ghana']

Processing abstract 3: Found countries: ['UK']

Processing abstract 4: Found countries: []

Processing abstract 5: Found countries: []

Processing abstract 6: Found countries: ['France', 'UK', 'Netherlands']

Processing abstract 7: Found countries: ['Ireland']

Processing abstract 8: Found countries: ['Canada']

Processing abstract 9: Found countries: ['UK']

Processing abstract 10: Found countries: ['Ireland']

Processing abstract 11: Found countries: ['Ireland']

Processing abstract 12: Found countries: ['UK']

Processing abstract 13: Found countries: ['Netherlands']

Processing abstract 14: Found countries: ['Britain', 'Germany']

Processing abstract 15: Found countries: ['UK']

Processing abstract 16: Found countries: ['Britain']

Processing abstract 17: Found countries: ['England']

Processing abstract 18: Found countries: []

Processing abstract 19: Found countries: [

In [30]:
# Optionally, convert this to a DataFrame and save to CSV

import pandas as pd
countries_df = pd.DataFrame(identified_countries)
countries_df['Article Title'] = titles.iloc[countries_df['abstract_index']]
countries_df['Abstract'] = abstracts.iloc[countries_df['abstract_index']]
countries_df.to_csv('abstracts_with_countries.csv', index=False)
print("\nResults saved to abstracts_with_countries.csv")
display(countries_df.head())


Results saved to abstracts_with_countries.csv


,abstract_index,abstract_snippet,countries,Article Title,Abstract
0,0,Was the European marriage pattern an important...,[England],Late marriage as a contributor to the industri...,Was the European marriage pattern an important...
1,1,The causes of large variation in the sizes of ...,"[Kenya, Ghana]",Late marriage and the HIV epidemic in sub-Saha...,The causes of large variation in the sizes of ...
2,2,A series of reforms to help low income familie...,[UK],WELFARE REFORM AND LONE PARENTS IN THE UK,A series of reforms to help low income familie...
3,3,The Labour government aims to increase the lon...,[],Lone parents and informal childcare: A tax cre...,The Labour government aims to increase the lon...
4,4,"Based on Norwegian register data, we show that...",[],Labor supply in the terminal stages of lone pa...,"Based on Norwegian register data, we show that..."


Any other model which is to be used can follow below. The evaluation can proceed with any number of models to gain confidence with the automated labeling or for any supervised/unsupervised tasks.

************End of Notebook***************
